## BirdCLEF 2026 - data download

The cell below downloads the competition data from Kaggle, but only if it is not already in the `data/` folder. Complete these steps once before running it.

**1. Verify your Kaggle account**
1. Sign in (or create an account) at [kaggle.com](https://www.kaggle.com).
2. Go to **Settings → Phone verification, Identity verification** and verify yourself. Kaggle blocks competition data downloads for unverified accounts.

**2. Accept the competition rules**
1. Open the [BirdCLEF 2026 competition](https://www.kaggle.com/competitions/birdclef-2026).
2. Click **Join Competition** / **I Understand and Accept**.

**3. Create a Kaggle API token**
1. Go to **Settings → API** on kaggle.com and create a new token (it starts with `KGAT_`).
2. Copy it right away - Kaggle shows it only once. Never share it or commit it to git.

**4. Make the token available to the notebook**
The token must be stored under the name `KAGGLE_API_TOKEN`. Use one of these options:

**Option A - Colab secrets (when running in Google Colab)**
1. Click the key icon (**Secrets**) in the left sidebar.
2. Add a new secret named `KAGGLE_API_TOKEN` with your token as the value.
3. Switch on **Notebook access** for it.

**Option B - `.env` file (local, VS Code, or Colab)**
1. Copy `.env.example` to `.env` and set `KAGGLE_API_TOKEN=your_token_here`.
2. Running in Colab: upload the `.env` file to the Colab working directory (**Files** panel → upload, usually `/content`). It is lost when the session ends, so upload it again each session.

**5. Run the cell**
Run the next cell. It downloads and extracts the data (~16 GB, so make sure there is enough free disk space). If the data is already there, it only prints the path.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COMPETITION = "birdclef-2026"
KAGGLE_TOKEN = "KAGGLE_API_TOKEN"  # name of the Colab secret / .env variable
DATA_ROOT = Path.cwd() / "data"  # git-ignored; kagglehub stores its cache here

# Files and folders that must exist for the data to count as complete
REQUIRED = [
    "train.csv",
    "taxonomy.csv",
    "train_soundscapes_labels.csv",
    "train_audio",
    "train_soundscapes",
]


def is_complete(folder):
    return all((folder / name).exists() for name in REQUIRED)


def find_local_data():
    """Return the folder holding the competition files, or None."""
    for train_csv in DATA_ROOT.rglob("train.csv"):
        if is_complete(train_csv.parent):
            return train_csv.parent
    return None


def load_from_colab_secrets():
    """Read the token from Colab Secrets; return False outside Colab."""
    try:
        from google.colab import userdata
    except ImportError:
        return False

    try:
        value = userdata.get(KAGGLE_TOKEN)
    except Exception:  # secret not defined or notebook access off
        return True
    if value:
        os.environ[KAGGLE_TOKEN] = value.strip()
    return True


def load_from_env_file():
    """Read the token from the nearest .env file (current folder or a parent)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if not env_file.is_file():
            continue
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip().removeprefix("export ").strip()
            value = value.strip().strip("\"'")
            if key == KAGGLE_TOKEN and value:
                os.environ[KAGGLE_TOKEN] = value
        print(f"Loaded .env from {env_file}")
        return True
    return False


DATA_DIR = find_local_data()

if DATA_DIR:
    print(f"Data already available: {DATA_DIR}")
else:
    print(f"Data not found in {DATA_ROOT}, downloading from Kaggle...")

    # Token: 1. Colab secrets, 2. .env file
    load_from_colab_secrets()
    if not os.environ.get(KAGGLE_TOKEN):
        load_from_env_file()
    if not os.environ.get(KAGGLE_TOKEN):
        raise EnvironmentError(
            f"{KAGGLE_TOKEN} not found. Add it as a Colab secret (notebook "
            f"access on) or to a .env file. Working directory: {Path.cwd()}"
        )

    # Keep the kagglehub cache inside the repo (must be set before the import).
    os.environ["KAGGLEHUB_CACHE"] = str(DATA_ROOT)

    try:
        import kagglehub
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
            check=True,
        )
        import kagglehub

    # Authenticate explicitly with the token
    from kagglehub.config import set_kaggle_api_token

    set_kaggle_api_token(os.environ[KAGGLE_TOKEN])

    # Fails here with a clear error if the token is rejected.
    print("Authenticated as:", kagglehub.whoami()["username"])

    # The competition rules must be accepted on kaggle.com (same account).
    DATA_DIR = Path(kagglehub.competition_download(COMPETITION))
    print("Path to competition files:", DATA_DIR)

    if not is_complete(DATA_DIR):
        missing = [n for n in REQUIRED if not (DATA_DIR / n).exists()]
        raise FileNotFoundError(f"Download finished but missing: {missing}")


Data already available: /content/data/competitions/birdclef-2026
